# 02 Descriptive Statistics

This notebook produces descriptive tables and figures for the final occupation-year panel.

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "outputs" / "figures"
TABLES = PROJECT_ROOT / "outputs" / "tables"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "final_dataset_ai_exposure_controls.csv")

df["post"] = (df["year"] >= 2023).astype(int)
df["high_ai"] = (df["ai_exposure"] > df["ai_exposure"].median()).astype(int)

print(df.shape)
df.head()

## Summary statistics

In [ ]:
summary_vars = [
    "relation", "log_relation", "vakanz_tage", "log_vakanz",
    "bestand_absolut", "log_bestand", "ai_exposure"
]

summary = (
    df[summary_vars]
    .describe()
    .T
    .round(3)
)

summary.to_csv(TABLES / "summary_statistics.csv")
summary

## AI exposure distribution

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.hist(df.drop_duplicates("kldb3")["ai_exposure"], bins=20, edgecolor="black")
plt.xlabel("AI exposure")
plt.ylabel("Number of occupations")
plt.title("Distribution of AI Exposure Across Occupations")
plt.tight_layout()
plt.savefig(FIGURES / "ai_exposure_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Descriptive trends by high and low AI exposure

In [ ]:
trend_relation = (
    df.groupby(["year", "high_ai"])["log_relation"]
    .mean()
    .reset_index()
)

trend_vakanz = (
    df.groupby(["year", "high_ai"])["log_vakanz"]
    .mean()
    .reset_index()
)

group_labels = {0: "Low AI exposure", 1: "High AI exposure"}

In [ ]:
plt.figure(figsize=(8, 5))

for group, label in group_labels.items():
    temp = trend_relation[trend_relation["high_ai"] == group]
    plt.plot(temp["year"], temp["log_relation"], marker="o", linewidth=2, label=label)

plt.axvline(2022, linestyle="--", linewidth=1)
plt.xlabel("Year")
plt.ylabel("Average log unemployment-to-vacancy ratio")
plt.title("Labour Market Slack by AI Exposure Group")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGURES / "descriptive_log_relation_by_ai_group.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for group, label in group_labels.items():
    temp = trend_vakanz[trend_vakanz["high_ai"] == group]
    plt.plot(temp["year"], temp["log_vakanz"], marker="o", linewidth=2, label=label)

plt.axvline(2022, linestyle="--", linewidth=1)
plt.xlabel("Year")
plt.ylabel("Average log vacancy duration")
plt.title("Vacancy Duration by AI Exposure Group")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGURES / "descriptive_log_vakanz_by_ai_group.png", dpi=300, bbox_inches="tight")
plt.show()